# Estimate Annual Grid Emissions Offset for GM-SEUS Arrays (eGRID + Cambium)
NOTE: Requires outputs from getGeneration (annual long generation table and 12x24 generation shapes).

This script estimates the operational grid carbon emissions offset (t CO2e) of every modeled GM-SEUS photovoltaic array for each operating year (installation year to 25 years after installation). For each array and calendar year, offset = annual generation (MWh) x effective grid emission rate (kg CO2e/MWh) / 1000.

**Grid emission rates**
* **Historical years (before `firstCambiumYear`)**: EPA eGRID annual total output emission rate (`SRC2ERTA`, lb CO2e/MWh) by eGRID subregion. eGRID is annual only, so historical rates are not weighted by time of generation. Years without an eGRID release are linearly interpolated.
* **Future years (`firstCambiumYear` onward)**: NLR Cambium 2024 hourly average emission rate of generation (`aer_gen_co2e_c`, kg CO2e/MWh) by Cambium generation and emission assessment (GEA) region. Cambium solve years are every 5 years (2025-2050); years between are linearly interpolated. Hourly rates are collapsed to month-by-hour (12x24) means.
* **Time-of-generation weighting (future years)**: each array's effective rate is its 12x24 generation shape (share of annual energy in each month-hour) multiplied by the 12x24 Cambium rate and summed. This captures whether an array generates when the grid is cleaner or dirtier than its annual average (e.g., midday summer in solar-heavy California vs. gas-peaking Michigan).

**Outputs**
* **GMSEUS_GridOffset_Annual_Long**: `arrayID`, `instYr`, `opYr`, `calYr`, `MWh`, `rateKgMWh`, `rateSource`, `tCO2e`.
* **GMSEUS_GridOffset_Annual_Wide**: `arrayID`, `co2Yr1` ... `co2Yr25` (t CO2e).
* **GMSEUS_GridRates_TimingCheck**: by region, the ratio of generation-weighted to flat annual Cambium rate (how much time of generation matters).

**Key assumptions** (all set as variables in the first code cells)
* Metric: average emission rate of in-region generation, direct combustion only (`aer_gen_co2e_c`), chosen to match eGRID total output rates (also combustion only, average). Marginal metrics (`srmer`, `lrmer`) can be swapped in for future years as a sensitivity; there is no matching historical marginal series in this script.
* Scenario: Cambium 2024 Mid-case. Other scenarios (e.g., low/high renewable energy cost) can be run by changing `cambiumScenario`.
* GWP: Cambium CO2e uses IPCC AR6 100-year GWPs; eGRID CO2e uses AR5. Grid CO2e is dominated by CO2, so the difference is small, but it is a known inconsistency.
* Geography: arrays are assigned to eGRID subregions (historical) and Cambium GEA regions (future) by centroid. This creates a seam at `firstCambiumYear`; the seam is plotted by region below.
* Operational phase only (no upstream manufacturing, construction, or end-of-life emissions).

# Import Libraries and Variables

In [ ]:
# Import libraries
import os
import re
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
temp_path = r'F:\BigPanel_Cache'
cambiumCache_path = os.path.join(temp_path, r'Cambium_Cache') # Cambium scenario zips (~850 MB each)
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
carbon_path = os.path.join(derived_path, r'carbonOffset')
generation_path = os.path.join(derived_path, r'generation') # getGeneration outputs
genWeatherSource = 'pvgis' # which getGeneration run to use ('pvgis' or 'nsrdb')
figure_path = os.path.join(wd, r'Figures')
grid_path = os.path.join(downloaded_path, r'GridEmissions')

# Set GM-SEUS array path (centroid latitude/longitude used for region assignment)
gmseusArraysPath = os.path.join(derived_path, r'GMSEUS_v2_1\SHP\GMSEUS_Arrays_Final_2026_v2_1.shp')

# Set getGeneration output paths (inputs here)
genLong_path = os.path.join(generation_path, r'GMSEUS_Generation_Annual_Long_' + genWeatherSource + '.parquet')
genShape_path = os.path.join(generation_path, r'GMSEUS_Generation_Shape12x24_' + genWeatherSource + '.parquet')

# Set region boundary paths
geaShp_path = os.path.join(grid_path, r'Cambium\gea_shapefiles') # folder from gea_shapefiles.zip (https://data.nlr.gov/submissions/289)
egridShp_path = os.path.join(grid_path, r'eGRID\eGRID_subregions') # folder from EPA eGRID mapping files (https://www.epa.gov/egrid/egrid-mapping-files)

# Set output paths
offsetLong_path = os.path.join(carbon_path, r'GMSEUS_GridOffset_Annual_Long_' + genWeatherSource + '.parquet')
offsetWide_path = os.path.join(carbon_path, r'GMSEUS_GridOffset_Annual_Wide_' + genWeatherSource + '.csv')
timingCheck_path = os.path.join(carbon_path, r'GMSEUS_GridRates_TimingCheck.csv')
rates12x24_path = os.path.join(carbon_path, r'Cambium_rates12x24.parquet')

# Make folders if they do not exist
os.makedirs(cambiumCache_path, exist_ok=True)
os.makedirs(os.path.join(grid_path, 'Cambium'), exist_ok=True)
os.makedirs(os.path.join(grid_path, 'eGRID'), exist_ok=True)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Cambium settings

# Cambium 2024 scenario zips (Catalyst Cooperative Zenodo archive of the NLR release, CC-BY-4.0)
cambiumZenodo = 'https://zenodo.org/records/18036226/files/'
cambiumScenario = 'mid_case' # e.g., 'low_renewable_energy_cost', 'high_renewable_energy_cost'
cambiumZipName = 'nrelcambium2024_' + cambiumScenario + '.zip'
cambiumZip_path = os.path.join(cambiumCache_path, cambiumZipName)

# Files inside the zip to use: GEA-level hourly files. VERIFY on first run with the zip listing cell below and adjust patterns.
cambiumFilePatterns = ['gea', 'hourly']

# Cambium metric (kg CO2e/MWh). Options: 'aer_gen_co2e_c' (default), 'aer_gen_co2e' (incl. precombustion), 'srmer_co2e_c', 'lrmer_co2e_c'
cambiumMetric = 'aer_gen_co2e_c'

# Cambium columns (from Cambium 2024 documentation): local standard time timestamp and GEA region
cambiumTimeCol = 'timestamp_local'
cambiumRegionCol = 'gea'

# First year taken from Cambium (Cambium solve years: 2025, 2030, ..., 2050)
firstCambiumYear = 2025

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ eGRID settings

# eGRID files by data year: (file path, subregion sheet name, header row). VERIFY sheet names and header rows for older releases.
# 2018-2023: EPA eGRID detailed data; 2005-2016: EPA historical eGRID zip; 2024: unofficial Cornerstone run of EPA's eGRID code (doi:10.5281/zenodo.18968658)
egridFiles = {
    2005: (os.path.join(grid_path, r'eGRID\eGRID2005_aggregation.xls'), 'SRL05', 4),
    2007: (os.path.join(grid_path, r'eGRID\eGRID2007_aggregation.xls'), 'SRL07', 4),
    2009: (os.path.join(grid_path, r'eGRID\eGRID2009_data.xls'), 'SRL09', 4),
    2010: (os.path.join(grid_path, r'eGRID\eGRID2010_Data.xls'), 'SRL10', 4),
    2012: (os.path.join(grid_path, r'eGRID\eGRID2012_Data.xlsx'), 'SRL12', 4),
    2014: (os.path.join(grid_path, r'eGRID\eGRID2014_Data_v2.xlsx'), 'SRL14', 1),
    2016: (os.path.join(grid_path, r'eGRID\egrid2016_data.xlsx'), 'SRL16', 1),
    2018: (os.path.join(grid_path, r'eGRID\egrid2018_data_v2.xlsx'), 'SRL18', 1),
    2019: (os.path.join(grid_path, r'eGRID\egrid2019_data.xlsx'), 'SRL19', 1),
    2020: (os.path.join(grid_path, r'eGRID\eGRID2020_Data_v2.xlsx'), 'SRL20', 1),
    2021: (os.path.join(grid_path, r'eGRID\eGRID2021_data.xlsx'), 'SRL21', 1),
    2022: (os.path.join(grid_path, r'eGRID\eGRID2022_data.xlsx'), 'SRL22', 1),
    2023: (os.path.join(grid_path, r'eGRID\egrid2023_data_rev2.xlsx'), 'SRL23', 1),
    2024: (os.path.join(grid_path, r'eGRID\egrid2024_data_metric.xlsx'), 'SRL24', 1), # Cornerstone (unofficial); metric file, see egridMetricYears
}

# eGRID columns: subregion code and CO2e total output emission rate. Older releases may lack SRC2ERTA; SRCO2RTA (CO2 only) is used as fallback.
egridRegionCol = 'SUBRGN'
egridRateCol = 'SRC2ERTA'
egridRateFallbackCol = 'SRCO2RTA'

# eGRID years whose file is in metric units (kg/MWh) rather than lb/MWh
egridMetricYears = [2024]
lbToKg = 0.45359237

# Use the unofficial eGRID2024? If False, 2024 is interpolated between eGRID2023 and Cambium 2025.
useEgrid2024Cornerstone = True

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Region boundary settings (VERIFY field names on first load)

geaNameCol = 'gea'
egridNameCol = 'Subregion'
regionCRS = 'EPSG:4269' # NAD83, matching GM-SEUS latitude/longitude

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Data conventions

noData = -9999
arrayIDcol = 'arrayID'
nYears = 25
shapeCols = ['m' + str(m).zfill(2) + 'h' + str(h).zfill(2) for m in range(1, 13) for h in range(24)]

# Helper Functions

In [ ]:
# A function to download a file with streaming (skips if already downloaded)
def downloadFile(url, outPath):
    if os.path.exists(outPath):
        print('Exists, skipping:', outPath)
        return outPath
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(outPath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print('Downloaded:', outPath)
    return outPath

# A function to read one Cambium csv from the zip. Metadata rows above the header are skipped by finding the header line.
def readCambiumCsv(zf, member):
    raw = zf.read(member).decode('utf-8', errors='ignore').splitlines()
    headerRow = next(i for i, line in enumerate(raw) if line.lower().startswith('timestamp'))
    df = pd.read_csv(io.StringIO('\n'.join(raw[headerRow:])), usecols=lambda c: c in [cambiumTimeCol, cambiumRegionCol, cambiumMetric])
    return df

# A function to read eGRID subregion rates for one year (returns region, year, rateKgMWh)
def readEgridYear(year, filePath, sheet, headerRow):
    df = pd.read_excel(filePath, sheet_name=sheet, header=headerRow)
    rateCol = egridRateCol if egridRateCol in df.columns else egridRateFallbackCol
    out = df[[egridRegionCol, rateCol]].dropna().copy()
    out.columns = ['egridRegion', 'rateKgMWh']
    out['rateKgMWh'] = out['rateKgMWh'].astype(float) * (1.0 if year in egridMetricYears else lbToKg)
    out['year'] = year
    out['rateCol'] = rateCol
    return out

# A function to assign each array point to a polygon region, with nearest-polygon fallback for points just outside (e.g., coastline)
def assignRegion(pointsGDF, regionsGDF, nameCol, outCol):
    joined = gpd.sjoin(pointsGDF, regionsGDF[[nameCol, 'geometry']], how='left', predicate='within')
    joined = joined[~joined.index.duplicated()]
    missing = joined[nameCol].isna()
    if missing.any():
        nearest = gpd.sjoin_nearest(pointsGDF[missing].to_crs('EPSG:6350'), regionsGDF[[nameCol, 'geometry']].to_crs('EPSG:6350'), how='left')
        nearest = nearest[~nearest.index.duplicated()]
        joined.loc[missing, nameCol] = nearest[nameCol]
    print(outCol, '| assigned by nearest polygon:', missing.sum())
    return joined[nameCol].rename(outCol)

# Assign Arrays to Grid Regions

In [ ]:
# Call GM-SEUS arrays and create centroid points from latitude/longitude
gmseusArrays = gpd.read_file(gmseusArraysPath)
arrayPts = gpd.GeoDataFrame(gmseusArrays[[arrayIDcol]].copy(), geometry=gpd.points_from_xy(gmseusArrays['longitude'], gmseusArrays['latitude']), crs=regionCRS)

# Call region boundaries and print columns (VERIFY geaNameCol and egridNameCol)
geaRegions = gpd.read_file(geaShp_path).to_crs(regionCRS)
egridRegions = gpd.read_file(egridShp_path).to_crs(regionCRS)
print('GEA columns:', list(geaRegions.columns), '| n =', len(geaRegions))
print('eGRID columns:', list(egridRegions.columns), '| n =', len(egridRegions))

In [ ]:
# Assign GEA region and eGRID subregion to each array
arrayRegions = arrayPts[[arrayIDcol]].copy()
arrayRegions['geaRegion'] = assignRegion(arrayPts, geaRegions, geaNameCol, 'geaRegion')
arrayRegions['egridRegion'] = assignRegion(arrayPts, egridRegions, egridNameCol, 'egridRegion')
arrayRegions = pd.DataFrame(arrayRegions)

# Print array counts by region
print(arrayRegions['geaRegion'].value_counts())
print(arrayRegions['egridRegion'].value_counts())

# Historical Rates: eGRID

In [ ]:
# Read every available eGRID year (skip 2024 if not using the unofficial release)
egridRates = []
for year, (filePath, sheet, headerRow) in egridFiles.items():
    if (year == 2024) and not useEgrid2024Cornerstone:
        continue
    if not os.path.exists(filePath):
        print('Missing eGRID file, skipping:', year, filePath)
        continue
    egridRates.append(readEgridYear(year, filePath, sheet, headerRow))
egridRates = pd.concat(egridRates, ignore_index=True)

# Print years read, the rate column used for each year, and a sanity summary (US subregion rates typically ~100-1,000 kg CO2e/MWh)
print(egridRates.groupby('year')['rateCol'].first())
print(egridRates.groupby('year')['rateKgMWh'].describe()[['count', 'min', '50%', 'max']].round(0))

# Future Rates: Cambium
## Download and Inspect

In [ ]:
# Download the Cambium scenario zip (~850 MB; skipped if present)
downloadFile(cambiumZenodo + cambiumZipName + '?download=1', cambiumZip_path)

# List zip contents and select GEA-level hourly csv files (VERIFY patterns against this listing)
cambiumZip = zipfile.ZipFile(cambiumZip_path)
allMembers = [m for m in cambiumZip.namelist() if m.lower().endswith('.csv')]
print('csv files in zip:', len(allMembers)); print(allMembers[:15])
cambiumMembers = [m for m in allMembers if all(p in m.lower() for p in cambiumFilePatterns)]
print('Selected files:', len(cambiumMembers)); print(cambiumMembers[:10])

## Collapse Hourly Rates to 12x24 by GEA Region and Solve Year

In [ ]:
# Loop through selected files, read metric, and average to month x hour by region and year
rates12x24 = []
for member in cambiumMembers:

    # Read csv; year taken from the timestamp (files may hold one or many regions)
    camDF = readCambiumCsv(cambiumZip, member)
    camDF['ts'] = pd.to_datetime(camDF[cambiumTimeCol])
    camDF['year'] = camDF['ts'].dt.year
    camDF['month'] = camDF['ts'].dt.month
    camDF['hour'] = camDF['ts'].dt.hour

    # If the region is not a column, take it from the file name (VERIFY)
    if cambiumRegionCol not in camDF.columns:
        camDF[cambiumRegionCol] = os.path.splitext(os.path.basename(member))[0]

    # Mean rate per region, year, month, hour
    rates12x24.append(camDF.groupby([cambiumRegionCol, 'year', 'month', 'hour'])[cambiumMetric].mean().reset_index())

# Combine and rename
rates12x24 = pd.concat(rates12x24, ignore_index=True).rename(columns={cambiumRegionCol: 'geaRegion', cambiumMetric: 'rateKgMWh'})
rates12x24 = rates12x24.groupby(['geaRegion', 'year', 'month', 'hour'], as_index=False)['rateKgMWh'].mean()

# Checks: solve years, regions, and every region-year has 288 cells
print('Solve years:', sorted(rates12x24['year'].unique()))
print('GEA regions:', rates12x24['geaRegion'].nunique())
assert (rates12x24.groupby(['geaRegion', 'year']).size() == 288).all()

# Check that array GEA names match Cambium names
print('Array GEA regions missing from Cambium:', set(arrayRegions['geaRegion'].dropna()) - set(rates12x24['geaRegion']))

# Export 12x24 rates
rates12x24.to_parquet(rates12x24_path)

# Effective Rate per Array and Year
For Cambium years, each array's effective rate is its generation-weighted 12x24 rate. For eGRID years, the annual rate is used as-is. Years between available data years are linearly interpolated per array; years beyond the last Cambium solve year are held at the last value.

In [ ]:
# Call generation outputs
genLong = pd.read_parquet(genLong_path)
genShape = pd.read_parquet(genShape_path)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Cambium: generation-weighted rate per array and solve year

# Pivot rates to wide 12x24 vectors per region and year (columns in the same order as shapeCols)
ratesWide = rates12x24.copy()
ratesWide['cell'] = 'm' + ratesWide['month'].astype(str).str.zfill(2) + 'h' + ratesWide['hour'].astype(str).str.zfill(2)
ratesWide = ratesWide.pivot_table(index=['geaRegion', 'year'], columns='cell', values='rateKgMWh')[shapeCols]

# Join array shapes to regions, then for each solve year compute sum(shape x rate)
shapeRegion = genShape.merge(arrayRegions[[arrayIDcol, 'geaRegion']], on=arrayIDcol, how='left')
cambiumEff = []
for year in sorted(rates12x24['year'].unique()):
    rY = ratesWide.xs(year, level='year')
    rMatched = rY.reindex(shapeRegion['geaRegion']).values
    cambiumEff.append(pd.DataFrame({arrayIDcol: shapeRegion[arrayIDcol], 'year': year,
                                    'rateKgMWh': np.nansum(shapeRegion[shapeCols].values * rMatched, axis=1),
                                    'rateFlatKgMWh': np.nanmean(rMatched, axis=1),
                                    'rateSource': 'Cambium2024_' + cambiumScenario}))
cambiumEff = pd.concat(cambiumEff, ignore_index=True)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ eGRID: annual rate per array and eGRID year (before firstCambiumYear)

egridEff = arrayRegions[[arrayIDcol, 'egridRegion']].merge(egridRates[egridRates['year'] < firstCambiumYear], on='egridRegion', how='inner')
egridEff = egridEff[[arrayIDcol, 'year', 'rateKgMWh']].assign(rateFlatKgMWh=egridEff['rateKgMWh'], rateSource='eGRID')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Interpolate to every calendar year needed by each array

# Anchor points per array: eGRID years + Cambium solve years
anchors = pd.concat([egridEff, cambiumEff], ignore_index=True).sort_values([arrayIDcol, 'year'])
anchorsByArray = {aid: grp for aid, grp in anchors.groupby(arrayIDcol)}

# For each array, interpolate the rate to its calendar years and label the source of each year
interpRows = []
for aid, calYrs in genLong.groupby(arrayIDcol)['calYr']:
    grp = anchorsByArray.get(aid)
    if grp is None:
        continue
    calYrs = calYrs.values
    rate = np.interp(calYrs, grp['year'].values, grp['rateKgMWh'].values) # flat beyond first/last anchor
    srcLookup = dict(zip(grp['year'], grp['rateSource']))
    src = [srcLookup.get(y, 'interpolated') for y in calYrs]
    interpRows.append(pd.DataFrame({arrayIDcol: aid, 'calYr': calYrs, 'rateKgMWh': rate, 'rateSource': src}))
arrayRates = pd.concat(interpRows, ignore_index=True)

# Label years beyond the last Cambium solve year
lastSolveYear = rates12x24['year'].max()
arrayRates.loc[arrayRates['calYr'] > lastSolveYear, 'rateSource'] = 'heldAtLastSolveYear'
print(arrayRates['rateSource'].value_counts())

# Calculate Annual Offsets and Build Output Tables

In [ ]:
# Join rates to generation and compute offset (t CO2e)
offsetLong = genLong.merge(arrayRates, on=[arrayIDcol, 'calYr'], how='left')
offsetLong['tCO2e'] = offsetLong['MWh'] * offsetLong['rateKgMWh'] / 1000

# Wide table: co2Yr1 ... co2YrN for every GM-SEUS array (non-modeled = -9999)
offsetWide = offsetLong.pivot(index=arrayIDcol, columns='opYr', values='tCO2e')
offsetWide.columns = ['co2Yr' + str(c) for c in offsetWide.columns]
offsetWide = gmseusArrays[[arrayIDcol]].merge(offsetWide.reset_index(), on=arrayIDcol, how='left').fillna(noData)

# Checks: no missing rates for modeled arrays, and totals
print('Rows with missing rate:', offsetLong['rateKgMWh'].isna().sum())
print('Total offset, opYr 1 (Mt CO2e):', round(offsetLong.loc[offsetLong['opYr'] == 1, 'tCO2e'].sum() / 1e6, 2))
print('Total offset, all', nYears, 'years (Mt CO2e):', round(offsetLong['tCO2e'].sum() / 1e6, 1))

# Checks: Time-of-Generation Effect and eGRID-Cambium Seam

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Timing check: generation-weighted vs flat annual Cambium rate by region (first solve year)

timingCheck = cambiumEff[cambiumEff['year'] == firstCambiumYear].merge(arrayRegions[[arrayIDcol, 'geaRegion']], on=arrayIDcol)
timingCheck = timingCheck.groupby('geaRegion')[['rateKgMWh', 'rateFlatKgMWh']].median()
timingCheck['weightedToFlat'] = timingCheck['rateKgMWh'] / timingCheck['rateFlatKgMWh']
print(timingCheck.round(3).sort_values('weightedToFlat'))
timingCheck.to_csv(timingCheck_path)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Seam check: median array rate by year and GEA region (look for jumps at firstCambiumYear)

seam = arrayRates.merge(arrayRegions[[arrayIDcol, 'geaRegion']], on=arrayIDcol).groupby(['geaRegion', 'calYr'])['rateKgMWh'].median().unstack('geaRegion')
fig, ax = plt.subplots(figsize=(10, 6))
seam.plot(ax=ax, legend=False, lw=1)
ax.axvline(firstCambiumYear, color='grey', ls='--')
ax.set_xlabel('Calendar year'); ax.set_ylabel('Median array emission rate (kg CO2e/MWh)')
ax.set_title('eGRID (historical) to Cambium (future) seam by GEA region')
fig.savefig(os.path.join(figure_path, 'gridRateSeam_byGEA.png'), dpi=200, bbox_inches='tight')
plt.show()

# Export

In [ ]:
# Export long and wide offsets
offsetLong.to_parquet(offsetLong_path)
offsetWide.round(2).to_csv(offsetWide_path, index=False)
print('Exported to:', carbon_path)